In [2]:
import requests

# ---------------- EXTRACT ----------------

url = "https://api.frankfurter.dev/v2/rates"

params = {
    "from": "2024-01-01",
    "to": "2026-09-29",
    "base": "EUR",
    "quotes": "USD,GBP,CHF,CNY"
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

print(type(data))
print("Number of records:", len(data))
print(data[:5])

StatementMeta(, 88ad6881-8b77-4493-885d-ba42af1e2dc1, 4, Finished, Available, Finished, False)

<class 'list'>
Number of records: 4012
[{'date': '2024-01-01', 'base': 'EUR', 'quote': 'CHF', 'rate': 0.92954}, {'date': '2024-01-01', 'base': 'EUR', 'quote': 'CNY', 'rate': 7.8599}, {'date': '2024-01-01', 'base': 'EUR', 'quote': 'GBP', 'rate': 0.86833}, {'date': '2024-01-01', 'base': 'EUR', 'quote': 'USD', 'rate': 1.1068}, {'date': '2024-01-02', 'base': 'EUR', 'quote': 'CHF', 'rate': 0.93134}]


In [3]:
import pandas as pd

# ---------------- TRANSFORM ----------------

df = pd.DataFrame(data)

df["date"] = pd.to_datetime(df["date"])

display(df)
print(df.dtypes)
print("Rows:", len(df))

StatementMeta(, 88ad6881-8b77-4493-885d-ba42af1e2dc1, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 48b7c1fa-d85d-4d21-97c5-16a63711afe6)

date     datetime64[ns]
base             object
quote            object
rate            float64
dtype: object
Rows: 4012


In [4]:
# ---------------- VALIDATE ----------------

print("Missing values:")
print(df.isna().sum())

print("\nDuplicate keys:")
print(df.duplicated(subset=["date", "base", "quote"]).sum())        #primary key uniqueness condition : No 2 rows with the same combo.

print("\nInvalid rates:")
print((df["rate"] <= 0).sum())

print("\nBase currencies:")
print(df["base"].unique())

print("\nQuote currencies:")
print(df["quote"].unique())

print("\nDate range:")
print(df["date"].min(), "→", df["date"].max())

StatementMeta(, 88ad6881-8b77-4493-885d-ba42af1e2dc1, 6, Finished, Available, Finished, False)

Missing values:
date     0
base     0
quote    0
rate     0
dtype: int64

Duplicate keys:
0

Invalid rates:
0

Base currencies:
['EUR']

Quote currencies:
['CHF' 'CNY' 'GBP' 'USD']

Date range:
2024-01-01 00:00:00 → 2026-09-29 00:00:00


In [5]:
# ---------------- PANDAS → SPARK ----------------

spark_df = spark.createDataFrame(df)

display(spark_df)
spark_df.printSchema()

StatementMeta(, 88ad6881-8b77-4493-885d-ba42af1e2dc1, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 26bae7aa-9928-4ab7-9869-0b059e033fe0)

root
 |-- date: timestamp (nullable = true)
 |-- base: string (nullable = true)
 |-- quote: string (nullable = true)
 |-- rate: double (nullable = true)



In [6]:
# ---------------- INITIAL HISTORICAL LOAD ----------------

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("exchange_rates")


check_df = spark.table("exchange_rates")

print("Rows:", check_df.count())
check_df.printSchema()
display(check_df)

StatementMeta(, 88ad6881-8b77-4493-885d-ba42af1e2dc1, 8, Finished, Available, Finished, False)

Rows: 4012
root
 |-- date: timestamp (nullable = true)
 |-- base: string (nullable = true)
 |-- quote: string (nullable = true)
 |-- rate: double (nullable = true)



SynapseWidget(Synapse.DataFrame, 6dd8c09a-5108-48d5-9dbe-4fe77d4e7d05)

In [7]:
spark.sql("SHOW TABLES").show()

StatementMeta(, 88ad6881-8b77-4493-885d-ba42af1e2dc1, 9, Finished, Available, Finished, False)

+--------------------+--------------+-----------+
|           namespace|     tableName|isTemporary|
+--------------------+--------------+-----------+
|`Exchange Rates D...|exchange_rates|      false|
+--------------------+--------------+-----------+

